# 05 — Diagnóstico de áreas sem setores válidos

A regra de distância escolhida foi atualizada para minimizar a soma total. Antes de recalcular L2, este notebook identifica se algum grupo de unidades ficou sem setores válidos atribuídos. Não excluímos grupos nem alteramos o denominador aqui: primeiro mostramos a situação para decidir como interpretar o ajuste.


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from scipy import stats

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PLANO.md").exists())
PROC = RAIZ / "dados" / "processados"
RETOMADA = PROC / "retomada"
DIST = RETOMADA / "distancias"
SAIDA = PROC / "analises" / "L2"
FIG = RAIZ / "figuras" / "L2"
SAIDA.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(RAIZ / "scripts"))
import estilo  # Apenas apresentação.
estilo.aplicar()
N_BOOT = 2000
SEMENTE = 42  # Mesma semente adotada na exploração anterior.

setores_brutos = {}
grupos = pd.read_csv(RETOMADA / "grupos_unidades.csv",
                     dtype={"CO_CNES_representante": str, "codigos_unidades": str})
cadastro_capitais = pd.read_csv(PROC / "capitais_resumo.csv")
def slug(nome):
    import unicodedata
    return unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode().split(" - ")[0].lower().replace(" ", "_")
nome_capital = dict(zip(cadastro_capitais.municipio.map(slug), cadastro_capitais.municipio))

def salvar_figura(figura, nome):
    figura.savefig(FIG / f"{nome}.png")
    figura.savefig(FIG / f"{nome}.pdf")

print("Bootstrap:", N_BOOT, "reamostragens por versão; semente:", SEMENTE)
print("Dados:", DIST.relative_to(RAIZ))


Bootstrap: 2000 reamostragens por versão; semente: 42
Dados: dados/processados/retomada/distancias


## 2. Ler os setores corrigidos e aplicar o filtro combinado

Mantemos setores com:

- população positiva;
- ligação ao nó da rua de até **2.000 m**;
- distância finita até o atendimento.

Não excluímos um setor da L2 porque sua renda está ausente: a renda não entra neste ajuste. Para a futura análise de renda, guardamos a média ponderada pela população com renda disponível e sua cobertura.

Se houver código de setor repetido, área não positiva ou grupo desconhecido, a execução para em vez de ajustar silenciosamente dados inconsistentes.


In [2]:
setores = {}
filtros = []
for versao, sufixo in [("SUS", "drive_sus"), ("Todos", "drive")]:
    arquivos = sorted(DIST.glob(f"*_{sufixo}_setores.csv"))
    assert len(arquivos) == 26, f"Esperados 26 arquivos: {versao}"
    partes = []
    for arquivo in arquivos:
        tabela = pd.read_csv(arquivo, dtype={"CD_SETOR": str, "CO_CNES": str})
        tabela["capital"] = nome_capital[arquivo.name.removesuffix(f"_{sufixo}_setores.csv")]
        partes.append(tabela)
    bruto = pd.concat(partes, ignore_index=True)
    setores_brutos[versao] = bruto
    assert not bruto.CD_SETOR.duplicated().any()
    mascara = (bruto.populacao > 0) & (bruto.ligacao_m <= 2000) & np.isfinite(bruto.d_m)
    valido = bruto.loc[mascara].copy()
    assert (valido.AREA_KM2 > 0).all() and np.isfinite(valido.AREA_KM2).all()
    setores[versao] = valido
    filtros.append({
        "versao": versao, "setores_no_arquivo": len(bruto), "setores_validos": len(valido),
        "populacao_habitada_antes_filtro": bruto.loc[bruto.populacao > 0].populacao.sum(),
        "populacao_valida": valido.populacao.sum(),
        "populacao_excluida": bruto.loc[bruto.populacao > 0].populacao.sum() - valido.populacao.sum(),
        "setores_validos_sem_renda": int(valido.renda_media.isna().sum()),
    })
display(pd.DataFrame(filtros).set_index("versao"))


,setores_no_arquivo,setores_validos,populacao_habitada_antes_filtro,populacao_valida,populacao_excluida,setores_validos_sem_renda
versao,,,,,,
SUS,92782,90985,45883071,45770048,113023,813
Todos,92782,90985,45883071,45770048,113023,813


## Identificar o que aconteceu com cada grupo

Comparamos todos os setores associados à unidade com os que permanecem depois dos filtros de população positiva, ligação à rua de até 2 km e distância finita. Assim distinguimos “nenhum setor foi associado” de “houve associação, mas todos os setores foram filtrados”.


In [3]:
diagnosticos = []
for versao in ["SUS", "Todos"]:
    bruto = setores_brutos[versao]
    valido = setores[versao]
    todos = bruto.groupby(["capital", "CO_CNES"]).agg(
        setores_associados=("CD_SETOR", "size"),
        moradores_associados=("populacao", "sum")
    ).reset_index()
    validos = valido.groupby(["capital", "CO_CNES"]).agg(
        setores_validos=("CD_SETOR", "size"),
        moradores_validos=("populacao", "sum")
    ).reset_index()
    cadastro = grupos.loc[grupos.versao == versao].rename(columns={"CO_CNES_representante": "CO_CNES"})
    d = cadastro.merge(todos, on=["capital", "CO_CNES"], how="left").merge(
        validos, on=["capital", "CO_CNES"], how="left")
    d[["setores_associados", "moradores_associados", "setores_validos", "moradores_validos"]] = d[[
        "setores_associados", "moradores_associados", "setores_validos", "moradores_validos"
    ]].fillna(0)
    d["motivo"] = np.select(
        [d.setores_associados == 0, d.setores_validos == 0],
        ["Nenhum setor associado", "Setores associados, mas todos saíram no filtro"],
        default="Tem setores válidos"
    )
    d["versao"] = versao
    diagnosticos.append(d)
diagnosticos = pd.concat(diagnosticos, ignore_index=True)
sem_area = diagnosticos.loc[diagnosticos.setores_validos == 0,
    ["versao", "capital", "codigos_unidades", "nomes_unidades", "n_unidades",
     "setores_associados", "moradores_associados", "motivo"]]
print("Resumo de grupos por situação:")
display(diagnosticos.groupby(["versao", "motivo"]).agg(
    grupos=("CO_CNES", "size"), unidades=("n_unidades", "sum"),
    setores_associados=("setores_associados", "sum"), moradores_associados=("moradores_associados", "sum")
).reset_index())
print("Grupos sem área válida para L2:")
display(sem_area)
SAIDA_DIAG = RETOMADA / "grupos_sem_setores_validos.csv"
sem_area.to_csv(SAIDA_DIAG, index=False)
print("Tabela registrada em:", SAIDA_DIAG.relative_to(RAIZ))

Resumo de grupos por situação:


,versao,motivo,grupos,unidades,setores_associados,moradores_associados
0,SUS,Tem setores válidos,245,245,92782,45883071
1,Todos,"Setores associados, mas todos saíram no filtro",1,1,2,1178
2,Todos,Tem setores válidos,272,273,92780,45881893


Grupos sem área válida para L2:


,versao,capital,codigos_unidades,nomes_unidades,n_unidades,setores_associados,moradores_associados,motivo
394,Todos,Salvador - BA,4445252,PA ILHA DE MARE,1,2,1178,"Setores associados, mas todos saíram no filtro"


Tabela registrada em: dados/processados/retomada/grupos_sem_setores_validos.csv


## Ver qual filtro removeu os setores

Para o grupo sem área, abrimos os dois setores associados e mostramos população, ligação à rua e distância calculada. Isso permite distinguir ausência de população, ligação longa e ausência de rota.


In [4]:
sem_codigos = sem_area[["versao", "capital", "codigos_unidades"]].drop_duplicates()
linhas = []
for registro in sem_codigos.itertuples(index=False):
    chave = slug(registro.capital)
    sufixo = "drive_sus" if registro.versao == "SUS" else "drive"
    arquivo = DIST / f"{chave}_{sufixo}_setores.csv"
    d = pd.read_csv(arquivo, dtype={"CD_SETOR": str, "CO_CNES": str})
    d = d.loc[d.CO_CNES == registro.codigos_unidades].copy()
    d["populacao_positiva"] = d.populacao > 0
    d["ligacao_ate_2km"] = d.ligacao_m <= 2000
    d["distancia_finita"] = np.isfinite(d.d_m)
    d["entra_no_filtro"] = d.populacao_positiva & d.ligacao_ate_2km & d.distancia_finita
    linhas.append(d[["capital", "CD_SETOR", "CO_CNES", "populacao", "ligacao_m", "ligacao_pa_m", "d_m",
                     "populacao_positiva", "ligacao_ate_2km", "distancia_finita", "entra_no_filtro"]])
detalhe_sem_area = pd.concat(linhas, ignore_index=True) if linhas else pd.DataFrame()
display(detalhe_sem_area)

,capital,CD_SETOR,CO_CNES,populacao,ligacao_m,ligacao_pa_m,d_m,populacao_positiva,ligacao_ate_2km,distancia_finita,entra_no_filtro
0,Salvador - BA,292740805100016,4445252,410,6135.714282,5166.370294,11302.084576,True,False,True,False
1,Salvador - BA,292740805100017,4445252,768,4904.090086,5166.370294,10070.460380,True,False,True,False


## Conferir ligações longas das unidades atribuídas

A ligação de um pronto-atendimento à rede também pode ser longa. Listamos unidades que foram escolhidas por algum setor que passa pelos filtros, para verificar se a nova regra está usando destinos ligados de forma suspeita.


In [5]:
atribuicoes = []
for versao, sufixo in [("SUS", "drive_sus"), ("Todos", "drive")]:
    for arquivo in sorted(DIST.glob(f"*_{sufixo}_setores.csv")):
        d = pd.read_csv(arquivo, dtype={"CD_SETOR": str, "CO_CNES": str})
        d = d.loc[(d.populacao > 0) & (d.ligacao_m <= 2000) & np.isfinite(d.d_m)].copy()
        if len(d):
            resumo = d.groupby(["capital", "CO_CNES"], as_index=False).agg(
                ligacao_pa_m=("ligacao_pa_m", "first"),
                setores_validos_atribuidos=("CD_SETOR", "size"),
                moradores_atribuidos=("populacao", "sum")
            )
            resumo["versao"] = versao
            atribuicoes.append(resumo)
atribuicoes = pd.concat(atribuicoes, ignore_index=True)
longas = atribuicoes.loc[atribuicoes.ligacao_pa_m > 2000].sort_values("ligacao_pa_m", ascending=False)
print("Unidades atribuídas a setores válidos cuja ligação à rede supera 2 km:")
display(longas)
print("Máxima ligação entre unidades que receberam setores válidos:", round(atribuicoes.ligacao_pa_m.max(), 1), "m")

Unidades atribuídas a setores válidos cuja ligação à rede supera 2 km:


,capital,CO_CNES,ligacao_pa_m,setores_validos_atribuidos,moradores_atribuidos,versao


Máxima ligação entre unidades que receberam setores válidos: 437.2 m
